## SQL-Fehleranalyse -Checkliste

Für AP01: die zehn Kernfragen prüfen

### Die zehn Kernfragen

1. Fehlen Objekte ohne Treffer? → `INNER JOIN` vs. `LEFT JOIN` prüfen; die vollständig benötigte Objekttabelle muss links stehen.
2. Sollte eine Gruppe 0 haben, zeigt aber 1? → Bei `LEFT JOIN` zählt `COUNT(*)` auch die ergänzte Zeile. Mit `COUNT(rechte_id)` nur echte Treffer zählen; dafür eine nicht nullable ID verwenden.
3. Gesucht sind „noch nie / ohne“? → `LEFT JOIN ... WHERE rechte_id IS NULL` oder `NOT EXISTS`; bei der NULL-Prüfung eine zuverlässig nicht nullable rechte ID verwenden.
4. Filter auf Einzelzeilen? → `WHERE`. Filter auf Gruppenkennzahlen? → `HAVING COUNT(...)`, `HAVING AVG(...)` usw.
5. Aggregation „pro X“? → `GROUP BY X`. Für einzelne Personen nach ID gruppieren, nicht nur nach Namen; nicht aggregierte Ausgabespalten grundsätzlich mitgruppieren.
6. Spalten aus mehreren Tabellen benötigt? → JOIN-Pfad über die passenden PK-/FK-Spalten prüfen; jede Verbindung braucht eine passende `ON`-Bedingung, ausser beim beabsichtigten `CROSS JOIN`.
7. Ungültige Kategorien gesucht? → `NOT IN ('gültig1', 'gültig2')`. Gross-/Kleinschreibung und Leerzeichen beachten; fehlende Kategorien bei Bedarf zusätzlich mit `OR spalte IS NULL` erfassen.
8. NULL geprüft? → `IS NULL` / `IS NOT NULL`, niemals `= NULL`. Ein leerer Text `''` ist kein NULL; auch `<>`, `NOT IN` und `NOT LIKE` wählen NULL nicht automatisch aus.
9. Sortierung gefordert? → `ORDER BY ... ASC/DESC`. Für `LIMIT`/`OFFSET` eine eindeutige Reihenfolge festlegen, z. B. `ORDER BY price DESC, bicycleid`; NULL-Platzierung bei Bedarf mit `NULLS LAST` bestimmen.
10. Ergebnis fachlich plausibel? → Zeilenanzahl, bekannte Objektanzahlen, Summen und 0-Fälle prüfen. Ein leeres Ergebnis kann korrekt sein.

### Wichtige Ergänzungen

11. Trotz `LEFT JOIN` fehlen Objekte ohne passende Verkäufe im Zeitraum? → Datumsfilter auf der rechten Tabelle in `ON` setzen, wenn alle linken Objekte erhalten bleiben sollen. Derselbe Filter in `WHERE` entfernt die NULL-Zeilen.
12. Summe fehlt statt 0? → `SUM`, `AVG`, `MIN` und `MAX` ignorieren NULL und liefern ohne bekannte Werte NULL. Falls fachlich passend: `COALESCE(SUM(betrag), 0)`; `COUNT(...)` liefert ohne Treffer 0.
13. Anzahl oder Umsatz zu hoch? → Mehrere 1:n-JOINs können Zeilen vervielfachen. JOIN-Kardinalitäten prüfen und nötigenfalls vor dem JOIN aggregieren. `COUNT(DISTINCT id)` zählt eindeutige Objekte; `SUM(DISTINCT betrag)` ist keine allgemeine Reparatur.
14. `NOT IN (SELECT ...)` liefert unerwartet keine Treffer? → NULL in der Unterabfrage ausschliessen oder `NOT EXISTS` verwenden. Bei NULL links ebenfalls die gewünschte Behandlung festlegen.
15. `AND` und `OR` kombiniert? → `AND` bindet stärker. Gewünschte Logik einklammern: `WHERE (make = 'Bianchi' OR make = 'Trek') AND price > 3000`.
16. Datum oder Preis an einer Grenze falsch gefiltert? → `BETWEEN` schliesst beide Grenzen ein. Für ganze Jahre mit Zeitstempeln besser `datum >= '2023-01-01' AND datum < '2024-01-01'` verwenden.
17. Textmuster verwechselt? → `LIKE`: `%` = beliebig viele Zeichen, `_` = genau eines; `ILIKE` ignoriert Gross-/Kleinschreibung. Regex: `~` / `~*`, `^` = Anfang, `$` = Ende. Python-Strings mit Regex-Backslashes als `r"""..."""` schreiben.
18. PostgreSQL meldet einen Typfehler bei `ROUND(..., 2)`? → Bei Gleitkommawerten zuerst umwandeln: `ROUND(CAST(AVG(price) AS NUMERIC), 2)`. Auch für `EXTRACT` muss die Quelle ein geeigneter Datums-/Zeittyp sein.
19. Alias oder Syntax wird nicht erkannt? → SELECT-Aliase in `ORDER BY` verwenden; in `WHERE` und `HAVING` den ursprünglichen Ausdruck schreiben. Kommas zwischen Spalten, keines direkt vor `FROM`; Textwerte mit `'...'`, Bezeichner bei Bedarf mit `"..."`.
20. Tabelle oder Python-Variable fehlt? → Richtige Datenbank und richtiges Schema prüfen; Imports, Verbindungsvariable und Datenimport ausführen. Eine Quelldatei im Repo ist noch keine PostgreSQL-Tabelle. Jeder Notebook-Kernel benötigt seine eigenen Python-Variablen; der gemeinsame Tabellenname `customers` gehört zum zuletzt geladenen Datensatz.
21. `UPDATE` oder `DELETE` betrifft zu viele Zeilen? → `WHERE` kontrollieren; ohne Bedingung werden alle Zeilen betroffen. Die Auswahl vorher mit `SELECT` und derselben Bedingung prüfen.

### SQL-Reihenfolge zum Nachschlagen

Schreibreihenfolge einer typischen Abfrage:

```sql
SELECT ...
FROM ...
LEFT JOIN ... ON ...
WHERE ...
GROUP BY ...
HAVING ...
ORDER BY ...
LIMIT ... OFFSET ...;
```

Logisch vereinfacht: `FROM/JOIN → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT/OFFSET`.

Merksatz: Erst die passenden Zeilen und Beziehungen bestimmen, dann gruppieren, anschliessend Gruppen filtern und das Ergebnis sortieren.